# Initial imports and definitions

In [8]:
# pip install rouge_score
from operator import itemgetter
import pandas as pd
from rouge_score import rouge_scorer

In [9]:
# Read the validation data
validation_df = pd.read_csv('data/validation.csv')
train_df = pd.read_csv('data/train.csv')
test_df = pd.read_csv('data/test_text.csv')

# Initialize Rouge Scorer
scorer = rouge_scorer.RougeScorer(['rougeL'])

In [3]:
validation_df

,text,titles
0,"Sur les réseaux sociaux, les images sont impre...","Le bateau de croisière, long de 275 m, a percu..."
1,La vidéo est devenue virale. Elle montre un po...,Le parquet de Paris a annoncé vendredi avoir o...
2,"Depuis la présidentielle, il est parfois un pe...","À Trappes (Yvelines), c'est désormais la star...."
3,"Routes endommagées, trains toujours perturbés,...",Un homme de 44 ans est porté disparu depuis sa...
4,Une enquête menée par le journal l'Obs.La nuit...,Son nom n'avait jusque-là jamais été cité dans...
...,...,...
1495,Plusieurs tags dont une croix gammée ont été r...,"Dans le Lot-et-Garonne, les fidèles musulmans ..."
1496,"Dylan Nacass, 23 ans, surfait à Bells Beach, u...",Un surfer français a survécu à une attaque de ...
1497,"""Pas possible... les Insoumis préfèrent le caf...","Florian Philippot, Jean-Luc Mélenchon et Laure..."
1498,Après l'annonce choc du géant japonais du pneu...,"De son côté, Laurent Berger a plaidé pour une ..."


# Lead-summary and EXT-ORACLE

In [128]:

# Function that generates summaries using LEAD-N
def lead_summary(text: pd.core.series.Series, titles: pd.core.series.Series, scorer: rouge_scorer.RougeScorer):
    summaries = []
    for idx, row in text.iteritems():
        sentences = row.split(".")
        summaries.append([idx, sentences[0] + "."])
    return summaries

# Function that generates summaries using EXT-ORACLE
def ext_oracle_summary(text: pd.core.series.Series, titles: pd.core.series.Series, scorer: rouge_scorer.RougeScorer):
    summaries = []
    for idx, row in text.items():
        sentences = row.split(".")
        reference = titles.iloc[idx]
        rs = [scorer.score(sentence, reference)['rougeL'][2] for sentence in sentences]
        index, element = max(enumerate(rs), key=itemgetter(1))
        summaries.append([idx, sentences[index]])  
    return summaries

In [ ]:
lead_summaries = lead_summary(validation_df['text'], validation_df['titles'], scorer)
ext_oracle_summaries = ext_oracle_summary(validation_df['text'], validation_df['titles'], scorer)

lead_rouge = []
ext_oracle_rouge = []
# Calculate the rouge-l score for each of the generated summaries compared to the original titles
for idx, title in validation_df['titles'].iteritems():
    lead_rouge.append(scorer.score(lead_summaries[idx][1], title)['rougeL'][2])
    ext_oracle_rouge.append(scorer.score(ext_oracle_summaries[idx][1], title)['rougeL'][2])

avg_rouge_score_lead = sum(lead_rouge) / len(lead_rouge)
avg_rouge_score_ext_oracle = sum(ext_oracle_rouge) / len(ext_oracle_rouge)

print("Average Rouge-L F-Score with LEAD-1: ", avg_rouge_score_lead)
print("Average Rouge-L F-Score with EXT-ORACLE:", avg_rouge_score_ext_oracle)

Extractive summarization to reduce the length of texts:

In [364]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def extractive_summarization(text, tokenizer, maxsize = 512):
    sentences = text.split('. ')  

    vectorizer = CountVectorizer().fit_transform(sentences)

    cosine_similarities = cosine_similarity(vectorizer, vectorizer)

    sentence_scores = np.zeros(len(sentences))
    for i in range(len(sentences)):
        for j in range(len(sentences)):
            if i != j:
                sentence_scores[i] += cosine_similarities[i][j]

    # Get the indices of the top N sentences with highest scores
    top_sentences_indices = np.argsort(sentence_scores)[-15:]
    top_sentences_indices.sort()
    i = 0
    summary = ""
    nb = 0
    while i<len(top_sentences_indices) and nb<maxsize:
        summary += " " + sentences[top_sentences_indices[i]]

        nb += len(tokenizer.encode(sentences[top_sentences_indices[i]], add_special_tokens=True))
        i+=1
    

    return summary[1:]

# Models from Hugging Face

The models we tested are :
- moussakam/barthez-orangesum-abstract
- mrm8488/camembert2camembert_shared-finetuned-french-summarization
- csebuetnlp/mT5_multilingual_XLSum
- bofenghuang/flan-t5-large-dialogsum-fr

In [11]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)

In [386]:
#modelname = "camembert2camembert_shared-finetuned-french-summarization"
#modelname = "bofenghuang/flan-t5-large-dialogsum-fr"
#modelname = "barthez-orangesum-abstract"
#modelname = "csebuetnlp/mT5_multilingual_XLSum"

modelname = "camembert_FT_10epoch"
#modelname = "finetuned_barthez"

tokenizer = AutoTokenizer.from_pretrained(modelname)
model = AutoModelForSeq2SeqLM.from_pretrained(modelname)

The following encoder weights were not tied to the decoder ['roberta/pooler']
The following encoder weights were not tied to the decoder ['roberta/pooler']
The following encoder weights were not tied to the decoder ['roberta/pooler']
The following encoder weights were not tied to the decoder ['roberta/pooler']


Visualization of the structure of the model

In [353]:
model

EncoderDecoderModel(
  (encoder): CamembertModel(
    (embeddings): CamembertEmbeddings(
      (word_embeddings): Embedding(32005, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): CamembertEncoder(
      (layer): ModuleList(
        (0-11): 12 x CamembertLayer(
          (attention): CamembertAttention(
            (self): CamembertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): CamembertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
            

Number of parameters of the model:

In [354]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

num_params = count_parameters(model)

print("Number of parameters:", num_params)

Number of parameters: 139612933


Test on one text

In [402]:
i=15
text_sentence = validation_df['text'][i]
word_count = len(text_sentence.split())

max_size = 512


input_ids = torch.tensor(
    [tokenizer.encode(text_sentence, add_special_tokens=True,truncation=True,max_length=512)]
) 

predict = model.generate(input_ids, max_length=512)[0]
s = tokenizer.decode(predict, skip_special_tokens=True, clean_up_tokenization_spaces=True)


text_sentence = extractive_summarization(text_sentence,tokenizer,maxsize=512)
input_ids = torch.tensor(
    [tokenizer.encode(text_sentence, add_special_tokens=True,truncation=True,max_length=512)]
) 
print(input_ids.size())
predict = model.generate(input_ids, max_length=512)[0]

print("True summary:")
print(validation_df['titles'][i])
print("\n")
print("Our summary : ")
print(s)
print("With extractive summary:")
print(tokenizer.decode(predict, skip_special_tokens=True, clean_up_tokenization_spaces=True))

torch.Size([1, 512])
True summary:
"La France est opposée, par principe, en tout temps et en tout lieu à la peine de mort", a rappelé le Quai d'Orsay.


Our summary : 
Trois Français ont déjà été reconnus coupables d'avoir rejoint l'EI en Irak.
With extractive summary:
Quatre Français ont déjà été reconnus coupables d'avoir rejoint l'organisation jihadiste État islamique en Irak.


In [403]:
print(scorer.score(s, validation_df['titles'][i])['rougeL'][2])
print(scorer.score(tokenizer.decode(predict, skip_special_tokens=True, clean_up_tokenization_spaces=True), validation_df['titles'][i])['rougeL'][2])


0.05
0.04651162790697675


In [265]:
len(tokenizer.encode(text_sentence, add_special_tokens=True,truncation=True,max_length=512))

512

In [377]:
def our_summary(text: pd.core.series.Series):
    summaries = []
    model.eval()
    max_size = 512 # Depends on the model chosen
    for idx, row in text.items():

        row = extractive_summarization(row,tokenizer, maxsize=512) #Extractive summary
        input_ids = torch.tensor(
            [tokenizer.encode(row,  add_special_tokens=True,truncation=True,max_length=512)]
        )

        #Concatenation of summaries
        """
        s = ""
        for i in range(input_ids.size(1)//max_size +1): # To overcome the model's limitations
            predict = model.generate(input_ids[:,i*max_size:min((i+1)*max_size,input_ids.size(1))], max_length=512)[0]
            s += " " + tokenizer.decode(predict, skip_special_tokens=True, clean_up_tokenization_spaces=True)
        """
        predict = model.generate(input_ids, max_length=1024)[0]
        summaries.append([idx, tokenizer.decode(predict, skip_special_tokens=True, clean_up_tokenization_spaces=True)])
    return summaries

Computation for $n$ texts

In [389]:
n = 150

my_summary_val = our_summary(validation_df['text'][:n])

In [301]:
my_summary_val

[[0,
  "Dimanche matin à Venise, l'équipage du MSC Opéra a perdu le contrôle du quai auquel il voulait s'arrimer. Quatre personnes ont été blessées dans l'accident."],
 [1,
  'Une enquête pour "violences volontaires par personne dépositaire de l\'autorité publique" a été ouverte après la diffusion d\'une vidéo montrant un policier tirant sur la foule lors d\'échauffourées à Paris.'],
 [2,
  "En campagne dans la quatrième circonscription de l'Essonne, Benoît Hamon est loin d'être assuré de ravir le fauteuil de délégué général de La République en Marche (LREM)."],
 [3,
  'Un pêcheur de 44 ans est porté disparu depuis samedi à Cassis (Bouches-du-Rhône), portant à trois le nombre de personnes recherchées.'],
 [4,
  'Selon les informations de l\'Obs, le "conseiller spécial" d\'Emmanuel Macron aurait transmis à l\'Élysée les images de la nuit qui a suivi les premières révélations du Monde.']]

In [137]:
# Test, combine with ext_oracle

our_summary_val_extOracle = ext_oracle_summary(validation_df['text'][:n], pd.Series([x[1] for x in my_summary_val]), scorer)


## Finetuning

In [26]:
torch.cuda.empty_cache()

In [ ]:
!python transformers/examples/pytorch/summarization/run_summarization.py \
    --model_name_or_path camembert_best_2000 \
    --do_train \
    --do_eval \
    --train_file data/train.csv \
    --validation_file data/validation_short.csv \
    --output_dir finetuned_barthez \
    --overwrite_output_dir \
    --per_device_train_batch_size=4 \
    --per_device_eval_batch_size=4 \
    --predict_with_generate \
    --fp16 \
    --learning_rate 1.4e-5\
    --max_source_length 512\
    --text_column text \
    --summary_column titles \
    --num_train_epochs 1 \
    --eval_steps 1000 \
    --evaluation_strategy steps \
    --save_strategy steps \
    --save_steps 1000

In [ ]:
%tensorboard --logdir logs/fit

In [ ]:
!python transformers/examples/pytorch/summarization/run_summarization.py \
    --model_name_or_path barthez-orangesum-abstract \
    --do_predict\
    --test_file data/validation_short.csv \
    --output_dir finetuned_barthez \
    --overwrite_output_dir \
    --per_device_train_batch_size=10 \
    --per_device_eval_batch_size=10 \
    --predict_with_generate \
    --fp16 \
    --max_source_length 1024\
    --text_column text \
    --summary_column titles \
    



In [65]:
with open("finetuned_barthez/generated_predictions.txt", "r") as file:
    sentences = file.read().split("\n")

# Remove any empty strings or whitespace
my_summary_val = [[i,sentence.strip()] for i,sentence in enumerate(sentences) if sentence.strip()]


In [66]:
my_summary_val

[[0,
  "Un paquebot de croisière a heurté un autre bateau touristique, dimanche matin à Venise. L'accident s'est produit dans le canal de la Giudecca, où de nombreux navires de croisière s'arrêtent pour permettre à leurs passagers de visiter la cité des Doges."],
 [1,
  'La vidéo montre un policier frappant à bout portant des manifestants avec un lanceur de balles de défense.'],
 [2,
  "Le candidat socialiste à la présidentielle est souvent moqué par les médias. En meeting, il explique qu'il ne sera pas candidat pour être délégué de classe."],
 [3,
  'Un homme est toujours porté disparu depuis samedi à Cassis (Bouches-du-Rhône), selon les premières informations de la gendarmerie.'],
 [4,
  "Selon les premières révélations du Monde, Alexandre Benalla aurait transmis à l'Élysée les images de l'affrontement entre un couple de manifestants et des policiers."],
 [5,
  "Le chef de l'Etat a annoncé lundi vouloir remettre en cause l'âge pivot et la réforme des retraites, mais a laissé planer l

# Test

In [405]:
rouge_score = []

for idx, title in validation_df['titles'][:n].items():
    generated_title = my_summary_val[idx][1]
    rouge_score.append(scorer.score(generated_title, title)['rougeL'][2])

avg_rouge_score = sum(rouge_score) / len(rouge_score)

print("Average Rouge-L F-Score: ", avg_rouge_score)

Average Rouge-L F-Score:  0.2415980049239032


Some results of the average Rouge-L score over the 50 first samples of the validation set:

- Camembert:   0.20720016979542005 ; fine tuned : 0.2488354478820943 (segmented : 0.21588701195212234, oracle : 0.20356866779680835, extractive : 0.2473937638335417)

- Barthez:  0.23063575394586303 

- Flan:  0.2001272719064669 

- t5 multilingual xl:  0.23236563920755748 

Rouge-L score over one sample:

In [113]:
print(scorer.score(ext_oracle_summaries[i][1], title)['rougeL'][2])
print(scorer.score(my_summary_val[i][1], title)['rougeL'][2])


0.2711864406779661
0.2711864406779661


# Apply on test set

In [339]:
my_summary_test = our_summary(test_df['text'])


Faster but less precise method:

In [ ]:
!python transformers/examples/pytorch/summarization/run_summarization.py \
    --model_name_or_path finetuned_barthez \
    --do_predict \
    --test_file data/test_text.csv \
    --output_dir finetuned_barthez \
    --overwrite_output_dir \
    --per_device_train_batch_size=4 \
    --per_device_eval_batch_size=4 \
    --predict_with_generate \
    --fp16 \
    --max_source_length 512\
    --text_column text \
    
    

In [33]:
with open("finetuned_barthez/generated_predictions.txt", "r") as file:
    sentences = file.read().split("\n")

# Remove any empty strings or whitespace
my_summary_test = [[i,sentence.strip()] for i,sentence in enumerate(sentences) if sentence.strip()]


# Save the result in `submission.csv`

In [340]:
submission_df = pd.DataFrame(my_summary_test, columns=['ID', 'titles'])
submission_df.to_csv('submission.csv', index=False)
